In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem

# 1. Load your collected PubChem data
input_file = "pubchem_bcl2_screening_set.csv"
print(f"Loading {input_file}...")
df = pd.read_csv(input_file)
df

Loading pubchem_bcl2_screening_set.csv...


,CID,ConnectivitySMILES
0,16196702,CN(C)C(CC1=CC=CC=C1)C(=O)C2=CC=CS2.Cl
1,1452362,CC(C)N1C(=NN=N1)SCC(=O)N2CCN(CC2)C(=O)C3=CC=CO3
2,4215262,CCC1=CC=C(C=C1)NC(=O)C(C)OC(=O)C2=C(N=CC=C2)OCC
3,2940489,CC1C=CCC2C1C(=O)N(C2=O)C3=CC=C(C=C3)OC(=O)C4=C...
4,3226716,C1CCC(CC1)NC(=O)NCCCN2CCCC2=O
...,...,...
6301,168969535,CC1=C(C=CC=C1OC)CN(C2=CC=C(C=C2)O)C(=O)C3=C(N(...
6302,168969605,CC1=C(C=CC=C1OC)CN(C2=CC3=C(C=C2)NN=C3)C(=O)C4...
6303,168969400,CC1=C(C=C(N1C)C2=C(C=CC(=C2)Cl)C(=O)N3CC4=CC=C...
6304,168969579,CC1=C(C=C(N1C)C2=C(C=CC(=C2)Cl)C(=O)N3CC4=CC=C...


In [5]:
df.isnull().sum()   


CID                   0
ConnectivitySMILES    0
dtype: int64

In [6]:

# Clear out any rows that are missing SMILES strings
df = df.dropna(subset=["ConnectivitySMILES"])


# 2. Define functions for Lipinski's Rule of Five (Ro5)
def calculate_lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:  # Handle invalid SMILES strings safely
        return None

    # Calculate the 4 classic Lipinski metrics
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)

    # Lipinski rule evaluation (allows max 1 violation)
    violations = 0
    if mw > 500:
        violations += 1
    if logp > 5:
        violations += 1
    if hbd > 5:
        violations += 1
    if hba > 10:
        violations += 1

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "Lipinski_Pass": violations <= 1,
    }


# 3. Calculate Fingerprints (Features for your ML Model)
def generate_morgan_fingerprint(smiles, radius=3, n_bits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # Generate bit vector
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    # Convert vector to a standard numpy array of 0s and 1s
    arr = np.zeros((0,), dtype=np.int8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr



In [8]:

# 4. Process the dataset row by row
print("Calculating descriptors and molecular fingerprints...")
lipinski_list = []
fingerprint_list = []
valid_indices = []

for idx, row in df.iterrows():
    smiles = row["ConnectivitySMILES"]

    lipinski = calculate_lipinski(smiles)
    fp_array = generate_morgan_fingerprint(smiles, radius=3, n_bits=1024)

    # Only keep the molecule if RDKit successfully parsed its structure
    if lipinski is not None and fp_array is not None:
        lipinski_list.append(lipinski)
        fingerprint_list.append(fp_array)
        valid_indices.append(idx)

# Re-align dataframe to discard failed rows
df_ready = df.iloc[valid_indices].copy()

# Merge Lipinski results back into the dataframe
df_lipinski = pd.DataFrame(lipinski_list, index=df_ready.index)
df_ready = pd.concat([df_ready, df_lipinski], axis=1)

# Separate the 1024 Morgan bit columns into an independent feature matrix (X)
X_features = np.array(fingerprint_list)
feature_cols = [f"Bit_{i}" for i in range(1024)]
df_features = pd.DataFrame(X_features, columns=feature_cols, index=df_ready.index)

# Combine text metadata, Lipinski filters, and features into one final spreadsheet
final_prepared_dataset = pd.concat([df_ready, df_features], axis=1)

# Save your final data structures
final_prepared_dataset.to_csv("pubchem_bcl2_ml_ready.csv", index=False)

print("\n--- Preparation Complete ---")
print(f"Total entries processed successfully: {final_prepared_dataset.shape[0]}")
print(
    f"Molecules passing Lipinski's Filter: {final_prepared_dataset['Lipinski_Pass'].sum()}"
)
print("File saved as 'pubchem_bcl2_ml_ready.csv'")
print(
    "Your features are now ready in columns 'Bit_0' through 'Bit_1023' for model prediction!"
)

Calculating descriptors and molecular fingerprints...


[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerator
[02:07:07] DEPRECATION WARNING: please use MorganGenerat


--- Preparation Complete ---
Total entries processed successfully: 6306
Molecules passing Lipinski's Filter: 1350
File saved as 'pubchem_bcl2_ml_ready.csv'
Your features are now ready in columns 'Bit_0' through 'Bit_1023' for model prediction!


In [11]:
final_prepared_dataset.shape

(6306, 1031)